# GRC Security Intelligence - Phase 1b: NAS share (physical security) synthetic data

Simulates the on-prem NAS share `\\acme-nas01\security` landed in the UC volume `grc_bronze.nas_share`:

| Path | Source | Format |
| --- | --- | --- |
| `acme-nas01/security/physical/badge_access/YYYY/MM/LenelS2_export_YYYYMMDD.csv` | LenelS2 badge readers (one file per weekday, site-local wall-clock time + IANA zone) | CSV |
| `acme-nas01/security/physical/visitor_log/YYYY/visitor_log_YYYY-Www.csv` | Visitor management weekly log | CSV |

Sites: HQ-NYC (America/New_York), SF-OFFICE (America/Los_Angeles), DAL-OPS and DC-1 (America/Chicago).

**Injected stories** (ground truth in `grc_synth.nas_story_truth`)
* S1 - 6 people terminated in the Sep-15 reorg (IdP orphans) keep badging in after termination; 3 of them also still sign in to Okta (Critical).
* S1b - 5 terminated employees try their badge after leaving and are correctly denied (ACCESS_DENIED_BADGE_DISABLED, 14 events).
* S2 - DC-1 server room `DC1-SR-01`: 5 denied attempts by unassigned contractor badge C0077812 at 01:51-01:59 on 2026-10-14, DOOR_FORCED_OPEN at 02:13, then 3 DOOR_HELD_OPEN (10-14, 10-15, 10-16).
* S3 - 3 walk-in vendor visits to DC-1 Server room 1 on 2026-10-15 without escort, NDA or sign-out.

Expected KPIs as of 2026-10-27: KPI-PHY-01 = 6, KPI-PHY-02 = 4, KPI-PHY-03 = 12, KPI-PHY-04 = 88.75%.

Requires `01_generate_synthetic_data` (reads `grc_synth.employees` and `grc_synth.identities`). Deterministic and idempotent: the NAS folder is rewritten on every run.

In [0]:
import os, random, datetime as dt
import pandas as pd
from pyspark.sql import functions as F, Window as W

dbutils.widgets.text("catalog", "workspace")
CAT = dbutils.widgets.get("catalog")
Q = f"`{CAT}`"
SYN = f"{Q}.grc_synth"
START, END, REORG = "2026-07-29", "2026-10-27", "2026-09-15"
LAST30_START = "2026-09-28"  # KPI window: visit_date > as_of_date - 30 days

spark.sql(f"CREATE VOLUME IF NOT EXISTS {Q}.grc_bronze.nas_share COMMENT 'Simulated on-prem NAS share \\\\acme-nas01\\security (Lakeflow Connect SFTP / NAS sync landing)'")
ROOT = f"/Volumes/{CAT}/grc_bronze/nas_share/acme-nas01/security/physical"

def u(col, salt):
    """Deterministic uniform [0,1) from a column + salt."""
    return F.pmod(F.hash(col, F.lit(salt)), F.lit(1000003)) / F.lit(1000003.0)

DOORS = [  # site_code, site_tz, door_id, door_name, zone, reader_serial
    ("HQ-NYC", "America/New_York", "HQ-L1-MAIN", "Lobby main turnstile", "Perimeter", "LNL-B9D729"),
    ("HQ-NYC", "America/New_York", "HQ-L4-EAST", "Floor 4 east", "Office", "LNL-AB9BA7"),
    ("HQ-NYC", "America/New_York", "HQ-L7-FIN", "Floor 7 finance suite", "Restricted", "LNL-843195"),
    ("SF-OFFICE", "America/Los_Angeles", "SF-L1-MAIN", "Main entrance", "Perimeter", "LNL-3D8888"),
    ("SF-OFFICE", "America/Los_Angeles", "SF-L2-OPEN", "Floor 2 open office", "Office", "LNL-F4EFC8"),
    ("DAL-OPS", "America/Chicago", "DAL-L1-MAIN", "Main entrance", "Perimeter", "LNL-3F15C3"),
    ("DAL-OPS", "America/Chicago", "DAL-NOC", "Network operations center", "Restricted", "LNL-0B9901"),
    ("DC-1", "America/Chicago", "DC1-MANTRAP", "Mantrap", "Perimeter", "LNL-0AA455"),
    ("DC-1", "America/Chicago", "DC1-SR-01", "Server room 1", "Critical", "LNL-E86D75"),
]
doors = spark.createDataFrame(DOORS, "site_code string, site_tz string, door_id string, door_name string, zone string, reader_serial string")
PERIM = {"HQ-NYC": "HQ-L1-MAIN", "SF-OFFICE": "SF-L1-MAIN", "DAL-OPS": "DAL-L1-MAIN"}
OFFICE = {"HQ-NYC": "HQ-L4-EAST", "SF-OFFICE": "SF-L2-OPEN"}

def door_for(mapping):
    e = None
    for k, v in mapping.items():
        e = F.when(F.col("site") == k, F.lit(v)) if e is None else e.when(F.col("site") == k, F.lit(v))
    return e

In [0]:
emp = (spark.table(f"{SYN}.employees")
       .join(spark.table(f"{SYN}.identities").select("employee_id", "is_orphan", "orphan_still_active"), "employee_id", "left")
       .withColumn("is_orphan", F.coalesce("is_orphan", F.lit(False)))
       .withColumn("orphan_still_active", F.coalesce("orphan_still_active", F.lit(False)))
       .withColumn("site", F.when(F.col("location").isin("New York", "Toronto", "London", "Dublin", "Berlin"), "HQ-NYC")
                          .when(F.col("location").isin("San Francisco", "Austin"), "SF-OFFICE")
                          .otherwise("DAL-OPS"))
       .withColumn("badge_id", F.concat(F.lit("B"), F.lpad(F.pmod(F.hash("employee_id", F.lit("badge")), F.lit(1000000)).cast("string"), 6, "0")))
       .withColumn("cardholder_name", F.concat_ws(" ", "first_name", "last_name"))
       .withColumn("cardholder_email", F.lower("email"))
       .withColumn("hire_date", F.col("hire_date").cast("date"))
       .withColumn("termination_date", F.col("termination_date").cast("date")))

# 25 Ops / Corp staff in Dallas hold DC-1 data-center access
dc_rank = W.orderBy(u(F.col("employee_id"), "dc"))
dc_staff = (emp.filter((F.col("site") == "DAL-OPS") & F.col("bu_id").isin("BU-OPS", "BU-CORP") & F.col("termination_date").isNull())
            .withColumn("r", F.row_number().over(dc_rank)).filter("r <= 25").select("employee_id", F.lit(True).alias("dc_access")))
emp = emp.join(dc_staff, "employee_id", "left").withColumn("dc_access", F.coalesce("dc_access", F.lit(False)))

# S1: 6 reorg orphans keep badging in (3 also still sign in to Okta -> Critical); no Finance staff, so no restricted-zone entries
o_rank = W.partitionBy("orphan_still_active").orderBy(u(F.col("employee_id"), "s1"))
s1 = (emp.filter(F.col("is_orphan") & (F.col("termination_date") == F.lit(REORG).cast("date")) & (F.col("bu_id") != "BU-FIN"))
      .withColumn("r", F.row_number().over(o_rank)).filter("r <= 3").drop("r"))

# S1b: 5 terminated (non-orphan) employees whose badge was correctly disabled
t_rank = W.orderBy(u(F.col("employee_id"), "s1b"))
s1b = (emp.filter(~F.col("is_orphan") & F.col("termination_date").between("2026-08-10", "2026-10-15"))
       .withColumn("r", F.row_number().over(t_rank)).filter("r <= 5").drop("r"))
display(s1.select("employee_id", "cardholder_email", "bu_id", "site", "orphan_still_active").union(
        s1b.select("employee_id", "cardholder_email", "bu_id", "site", "orphan_still_active")))

In [0]:
days = (spark.sql(f"SELECT explode(sequence(DATE'{START}', DATE'{END}')) AS d")
        .filter(~F.dayofweek("d").isin(1, 7)))

def at(day_col, start_h, span_h, key, salt):
    secs = (F.lit(start_h * 3600) + u(key, salt) * F.lit(span_h * 3600)).cast("long")
    return F.timestamp_seconds(F.unix_timestamp(F.col(day_col).cast("timestamp")) + secs)

COLS = ["ts", "door_id", "badge_id", "cardholder_name", "cardholder_email", "cardholder_type", "event_type"]
key = F.concat_ws("|", "employee_id", F.col("d").cast("string"))

# Regular workforce presence: badge valid from hire date through termination date, ~55% of weekdays on site
present = (emp.crossJoin(days)
           .filter((F.col("d") >= F.col("hire_date")) & (F.col("termination_date").isNull() | (F.col("d") <= F.col("termination_date"))))
           .filter(u(key, "present") < 0.55)
           .withColumn("t0", at("d", 7, 3, key, "t_in"))
           .withColumn("cardholder_type", F.lit("Employee")).withColumn("event_type", F.lit("ACCESS_GRANTED")))

perimeter = present.select(F.col("t0").alias("ts"), door_for(PERIM).alias("door_id"), *COLS[2:])
office = (present.filter(F.col("site").isin(*OFFICE) & (u(key, "office") < 0.55))
          .select((F.col("t0") + F.expr("INTERVAL 5 MINUTES") + (u(key, "o_gap") * 35).cast("int") * F.expr("INTERVAL 1 MINUTE")).alias("ts"),
                  door_for(OFFICE).alias("door_id"), *COLS[2:]))
restricted = (present.filter(((F.col("site") == "HQ-NYC") & (F.col("bu_id") == "BU-FIN")) |
                             ((F.col("site") == "DAL-OPS") & F.col("bu_id").isin("BU-OPS", "BU-CORP")))
              .filter(u(key, "restricted") < 0.8)
              .select(at("d", 10, 7, key, "t_r").alias("ts"),
                      F.when(F.col("site") == "HQ-NYC", "HQ-L7-FIN").otherwise("DAL-NOC").alias("door_id"), *COLS[2:]))
dc_days = present.filter(F.col("dc_access") & (u(key, "dc_day") < 0.09)).withColumn("tdc", at("d", 10, 5, key, "t_dc"))
dc_mantrap = dc_days.select(F.col("tdc").alias("ts"), F.lit("DC1-MANTRAP").alias("door_id"), *COLS[2:])
dc_sr = dc_days.select((F.col("tdc") + F.expr("INTERVAL 3 MINUTES")).alias("ts"), F.lit("DC1-SR-01").alias("door_id"), *COLS[2:])

# S1: orphans keep badging in after the Sep-15 termination (perimeter + office only)
s1_days = (s1.crossJoin(days.filter(F.col("d") > F.lit(REORG).cast("date")))
           .filter(u(key, "s1_day") < 0.35)
           .withColumn("t0", at("d", 7, 3, key, "s1_t"))
           .withColumn("cardholder_type", F.lit("Employee")).withColumn("event_type", F.lit("ACCESS_GRANTED")))
s1_ev = s1_days.select(F.col("t0").alias("ts"), door_for(PERIM).alias("door_id"), *COLS[2:]).unionByName(
        s1_days.filter(F.col("site").isin(*OFFICE) & (u(key, "s1_office") < 0.4))
               .select((F.col("t0") + F.expr("INTERVAL 12 MINUTES")).alias("ts"), door_for(OFFICE).alias("door_id"), *COLS[2:]))

# S1b: denied attempts with a disabled badge after termination (4+3+3+2+2 = 14 events)
rows = []
for n, r in zip([4, 3, 3, 2, 2], s1b.orderBy("employee_id").collect()):
    for k in range(n):
        d = r.termination_date + dt.timedelta(days=1 + 2 * k)
        d += dt.timedelta(days={5: 2, 6: 1}.get(d.weekday(), 0))
        rows.append((dt.datetime.combine(d, dt.time(8, 4 + 7 * k, 17)), PERIM[r.site], r.badge_id, r.cardholder_name,
                     r.cardholder_email, "Employee", "ACCESS_DENIED_BADGE_DISABLED"))
# S2: DC-1 server room break-in sequence
for t in ["01:51:04", "01:52:40", "01:55:12", "01:58:03", "01:59:47"]:
    rows.append((dt.datetime.fromisoformat(f"2026-10-14 {t}"), "DC1-SR-01", "C0077812", "Unassigned contractor badge", None,
                 "Contractor", "ACCESS_DENIED_NO_CLEARANCE"))
for t, e in [("2026-10-14 02:13:45", "DOOR_FORCED_OPEN"), ("2026-10-14 02:31:10", "DOOR_HELD_OPEN"),
             ("2026-10-15 23:08:10", "DOOR_HELD_OPEN"), ("2026-10-16 03:44:10", "DOOR_HELD_OPEN")]:
    rows.append((dt.datetime.fromisoformat(t), "DC1-SR-01", None, None, None, "Unknown", e))
story = spark.createDataFrame(rows, "ts timestamp, door_id string, badge_id string, cardholder_name string, cardholder_email string, cardholder_type string, event_type string")

ev = (perimeter.unionByName(office).unionByName(restricted).unionByName(dc_mantrap).unionByName(dc_sr)
      .unionByName(s1_ev).unionByName(story)
      .join(doors, "door_id")
      .withColumn("seq", F.row_number().over(W.orderBy("ts", "door_id", "badge_id")))
      .withColumn("event_id", F.concat(F.lit("EVT"), F.date_format("ts", "yyyyMMdd"), F.lpad(F.col("seq").cast("string"), 7, "0")))
      .withColumn("event_time", F.date_format("ts", "yyyy-MM-dd HH:mm:ss"))
      .withColumn("file_date", F.to_date("ts"))
      .select("event_id", "event_time", "site_tz", "site_code", "door_id", "door_name", "zone", "badge_id", "cardholder_name",
              "cardholder_email", "cardholder_type", "event_type", "reader_serial", "file_date"))
badge_pdf = ev.toPandas()
print(len(badge_pdf), "badge events")
display(ev.groupBy("site_code", "door_id", "event_type").count().orderBy("site_code", "door_id"))

In [0]:
# Visit counts by (site, area, unescorted, nda, not_signed_out, in_last_30_days). The DC-1 unescorted rows are story S3.
DIST = [
    ("DAL-OPS", "Lobby", 0, "N", 0, 0, 3), ("DAL-OPS", "Lobby", 0, "N", 0, 1, 1), ("DAL-OPS", "Lobby", 0, "Y", 0, 0, 43),
    ("DAL-OPS", "Lobby", 0, "Y", 0, 1, 22), ("DAL-OPS", "Network operations center", 0, "N", 0, 0, 1),
    ("DAL-OPS", "Network operations center", 0, "Y", 0, 0, 24), ("DAL-OPS", "Network operations center", 0, "Y", 0, 1, 15),
    ("DC-1", "Mantrap", 0, "N", 0, 0, 1), ("DC-1", "Mantrap", 0, "N", 0, 1, 2), ("DC-1", "Mantrap", 0, "Y", 0, 0, 42),
    ("DC-1", "Mantrap", 0, "Y", 0, 1, 17), ("DC-1", "Server room 1", 0, "Y", 0, 0, 44), ("DC-1", "Server room 1", 0, "Y", 0, 1, 25),
    ("DC-1", "Server room 1", 1, "N", 1, 1, 3),
    ("HQ-NYC", "Floor 4 meeting rooms", 0, "Y", 0, 0, 12), ("HQ-NYC", "Floor 4 meeting rooms", 0, "Y", 0, 1, 5),
    ("HQ-NYC", "Floor 4 meeting rooms", 1, "Y", 0, 0, 5), ("HQ-NYC", "Floor 4 meeting rooms", 1, "Y", 0, 1, 4),
    ("HQ-NYC", "Floor 7 finance suite", 0, "Y", 0, 0, 11), ("HQ-NYC", "Floor 7 finance suite", 0, "Y", 0, 1, 5),
    ("HQ-NYC", "Floor 7 finance suite", 1, "N", 0, 0, 1), ("HQ-NYC", "Floor 7 finance suite", 1, "Y", 0, 0, 11),
    ("HQ-NYC", "Floor 7 finance suite", 1, "Y", 0, 1, 9), ("HQ-NYC", "Lobby", 0, "N", 0, 1, 2), ("HQ-NYC", "Lobby", 0, "Y", 0, 0, 8),
    ("HQ-NYC", "Lobby", 0, "Y", 0, 1, 8), ("HQ-NYC", "Lobby", 1, "N", 0, 0, 1), ("HQ-NYC", "Lobby", 1, "N", 0, 1, 1),
    ("HQ-NYC", "Lobby", 1, "Y", 0, 0, 11), ("HQ-NYC", "Lobby", 1, "Y", 0, 1, 9), ("SF-OFFICE", "Floor 2", 0, "Y", 0, 0, 20),
    ("SF-OFFICE", "Floor 2", 0, "Y", 0, 1, 7), ("SF-OFFICE", "Floor 2", 1, "N", 0, 0, 1), ("SF-OFFICE", "Floor 2", 1, "Y", 0, 0, 23),
    ("SF-OFFICE", "Floor 2", 1, "Y", 0, 1, 12), ("SF-OFFICE", "Lobby", 0, "Y", 0, 0, 20), ("SF-OFFICE", "Lobby", 0, "Y", 0, 1, 8),
    ("SF-OFFICE", "Lobby", 1, "N", 0, 0, 1), ("SF-OFFICE", "Lobby", 1, "Y", 0, 0, 19), ("SF-OFFICE", "Lobby", 1, "Y", 0, 1, 5),
]
rng = random.Random(20261027)
weekdays = [dt.date.fromisoformat(START) + dt.timedelta(days=i) for i in range(91)]
weekdays = [d for d in weekdays if d.weekday() < 5 and d <= dt.date.fromisoformat(END)]
recent = [d for d in weekdays if d >= dt.date.fromisoformat(LAST30_START)]
older = [d for d in weekdays if d < dt.date.fromisoformat(LAST30_START)]

hosts = {}
for r in emp.filter(F.col("termination_date").isNull()).select("site", "cardholder_email", "dc_access").collect():
    hosts.setdefault(r.site, []).append(r.cardholder_email)
    if r.dc_access:
        hosts.setdefault("DC-1", []).append(r.cardholder_email)
for k in hosts:
    hosts[k].sort()

FIRST = ["Sam", "Taylor", "Avery", "Jordan", "Casey", "Morgan", "Riley", "Quinn", "Jamie", "Drew", "Alex", "Robin", "Skyler", "Parker", "Reese"]
LAST = ["Smith", "Nguyen", "Garcia", "Patel", "Kim", "Brown", "Lopez", "Chen", "Walker", "Young", "Hall", "Rivera", "Scott", "Green", "Adams"]
COMPANY = {"DC-1": ["Iron Mountain", "Equinix Services", "Schneider Electric", "Vertiv", "Dell Technologies"],
           "DAL-OPS": ["Cisco", "Johnson Controls", "Grainger", "CDW", "Accenture"],
           "HQ-NYC": ["Deloitte", "KPMG", "Goldman Sachs", "Bloomberg", "PwC", "Recruiting candidate"],
           "SF-OFFICE": ["Salesforce", "Okta", "Johnson Controls", "Snowflake", "Recruiting candidate"]}

visits = []
for site, area, unesc, nda, no_out, last30, n in DIST:
    for _ in range(n):
        story_s3 = site == "DC-1" and unesc == 1
        d = dt.date(2026, 10, 15) if story_s3 else rng.choice(recent if last30 else older)
        host = rng.choice(hosts.get(site) or hosts["DAL-OPS"])
        t_in = dt.datetime.combine(d, dt.time(9)) + dt.timedelta(minutes=rng.randint(0, 360))
        t_out = t_in + dt.timedelta(minutes=rng.randint(15, 180))
        visits.append({
            "visit_date": d.isoformat(), "site_code": site,
            "visitor_name": f"{rng.choice(FIRST)} {rng.choice(LAST)}",
            "visitor_company": "Walk-in vendor (no PO)" if story_s3 else rng.choice(COMPANY[site]),
            "host_email": host, "escort_email": "" if unesc else host, "area_visited": area,
            "sign_in_time": t_in.strftime("%H:%M"), "sign_out_time": "" if no_out else t_out.strftime("%H:%M"),
            "nda_signed": nda, "_sort": (d, t_in)})
visits.sort(key=lambda v: v["_sort"])
for i, v in enumerate(visits, 1):
    v["visit_id"] = f"V{i:06d}"
visit_pdf = pd.DataFrame(visits)[["visit_id", "visit_date", "site_code", "visitor_name", "visitor_company", "host_email",
                                  "escort_email", "area_visited", "sign_in_time", "sign_out_time", "nda_signed"]]
print(len(visit_pdf), "visits")

In [0]:
dbutils.fs.rm(ROOT, True)

badge_cols = ["event_id", "event_time", "site_tz", "site_code", "door_id", "door_name", "zone", "badge_id", "cardholder_name",
              "cardholder_email", "cardholder_type", "event_type", "reader_serial"]
for d, g in badge_pdf.groupby("file_date"):
    folder = f"{ROOT}/badge_access/{d:%Y}/{d:%m}"
    os.makedirs(folder, exist_ok=True)
    g.sort_values("event_id")[badge_cols].to_csv(f"{folder}/LenelS2_export_{d:%Y%m%d}.csv", index=False)

visit_pdf["_wk"] = pd.to_datetime(visit_pdf["visit_date"]).dt.isocalendar().apply(lambda r: f"{r.year}-W{r.week:02d}", axis=1)
for wk, g in visit_pdf.groupby("_wk"):
    folder = f"{ROOT}/visitor_log/{wk[:4]}"
    os.makedirs(folder, exist_ok=True)
    g.drop(columns="_wk").to_csv(f"{folder}/visitor_log_{wk}.csv", index=False)

truth = ([("S1_badge_after_termination", r.cardholder_email, "orphan badge still active; ACCESS_GRANTED after 2026-09-15" +
           (" (also signs in to Okta)" if r.orphan_still_active else "")) for r in s1.collect()] +
         [("S1b_denied_after_termination", r.cardholder_email, "terminated user badge correctly disabled") for r in s1b.collect()] +
         [("S2_dc1_forced_door", "DC1-SR-01", "5 denied contractor attempts 01:51-01:59, DOOR_FORCED_OPEN 02:13 on 2026-10-14, 3 DOOR_HELD_OPEN 10-14..16"),
          ("S3_unescorted_dc_visitors", "DC-1", "3 walk-in vendor visits to Server room 1 on 2026-10-15 without escort or NDA, no sign-out")])
(spark.createDataFrame(truth, "story_id string, subject string, description string")
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SYN}.nas_story_truth"))

n_badge = len(dbutils.fs.ls(f"{ROOT}/badge_access/2026/10"))
print(f"Wrote {len(badge_pdf)} badge events ({badge_pdf.file_date.nunique()} daily files) and {len(visit_pdf)} visits "
      f"({visit_pdf._wk.nunique()} weekly files) to {ROOT}")